In [ ]:
import pandas as pd
import requests
from io import StringIO

# 下載 OCC 每週選擇權 CSV
import pandas as pd
import requests
from io import StringIO

url = "https://marketdata.theocc.com/weekly-options?action=download"
response = requests.get(url)
csv_text = response.content.decode("utf-8")

# 讀取資料
df = pd.read_csv(StringIO(csv_text), skiprows=6)
left = df.iloc[:, 0].dropna().astype(str).str.strip().tolist()
right = df.iloc[:, 4].dropna().astype(str).str.strip().tolist()
common_tickers = sorted(set(left) & set(right))

# 特殊處理 BRKB -> BRK-B
processed_tickers = ["BRK-B" if t.upper() == "BRKB" else t for t in common_tickers]

# 儲存為 weekly_option.txt
with open("weekly_option.txt", "w") as f:
    for ticker in processed_tickers:
        f.write(f"{ticker}\n")

# 顯示前 X 筆供確認
processed_tickers[:10]


# 顯示前 X 個 ticker 做確認
print("✅ 已儲存至 /content/weekly_option.txt，前10筆如下：")
print(common_tickers[:10])


In [ ]:
import datetime
import yfinance as yf

import pandas as pd

# 讀取已處理好的 symbol 清單
with open('weekly_option.txt', 'r') as f:
    tickers = [line.strip().upper() for line in f if line.strip()]

# 確認前幾筆 ticker
print(tickers[:10])


filtered = []

for symbol in tickers:
    t = yf.Ticker(symbol)
    info = t.info
    # X. Price < $X
    price = info.get('regularMarketPrice', 0)
    if price is None or price >= 40:
        continue

    # X. Get dividend history and check only the latest payout > $X
    div_hist = t.dividends  # pandas Series sorted by date
    if div_hist.empty:
        continue
    last_div = div_hist.iloc[-1]  # most recent dividend amount
    if last_div <= 0.1:
        continue

    # X. Annual dividend yield ≥ X%
    #    Try info['dividendYield']; fall back to info['dividendRate']/price if needed
    div_yield = info.get('dividendYield')
    # print(div_yield)
    if not div_yield:
        continue
    if div_yield < 1:
        continue

    # X. Weekly options check: any back-to-back expirations ≤ X days apart
    exps = t.options
    if not exps:
        continue
    dates = sorted(datetime.datetime.strptime(d, '%Y-%m-%d') for d in exps)
    gaps = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
    if not any(g <= 8 for g in gaps):
        continue

    ex_ts = info.get("exDividendDate")   # UNIX timestamp
    if not ex_ts or ex_ts == 0:
        continue
    next_ex = datetime.datetime.fromtimestamp(ex_ts).date()
    today = datetime.date.today()
    if next_ex <= today:
        continue


    filtered.append([symbol, price, div_yield, next_ex])

print(f"{'TICKER':6} {'PRICE':>7} {'YIELD':>7} {'EX-DIVIDEND'}")
for i in filtered:
    print(f"{i[0]:6} | ${i[1]:6.2f} | {i[2]:5.1f}% | {i[3]}")

    # X. Passed all filters—collect
    # filtered.append({
    #     'ticker': symbol,
    #     'price': price,
    #     'payouts_per_year': div_freq,
    #     'dividend_yield_pct': div_yield * X,
    #     'next_expirations': exps[:X]
    # })

# # X. Display results
# print(f"{'TICKER':X} {'PRICE':>X} {'YIELD':>X} {'PAYOUTS':>X} {'EXPIRATIONS'}")
# for s in filtered:
#     print(f"{s['ticker']:X} "
#           f"${s['price']:Xf} "
#           f"{s['dividend_yield_pct']:Xf}% "
#           f"{s['payouts_per_year']:Xd}   "
#           f"{', '.join(s['next_expirations'])}")
